# 03 — Flat Regional Map (PlateCarrée) and World Map (Robinson)

Two flat projections commonly used for regional and global coverage — built entirely from scratch.

---
**Stack used:** `noawclg`, `xarray`, `numpy`, `matplotlib`, `cartopy`, `cmocean`

In [ ]:
import noawclg
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cmocean
import copy

In [ ]:
# Dark theme constants
BG = "#0d1117"; LAND = "#13171d"; OCEAN = "#0d1520"
COAST = "#2d6db5"; BORDER = "#2a2f36"; STATE = "#1e2530"
TEXT = "#e6edf3"; SUBTEXT = "#8b949e"
BOXBG = "#161b22"; BOXEDGE = "#30363d"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "axes.labelcolor":   TEXT,
    "xtick.color":       SUBTEXT,
    "ytick.color":       SUBTEXT,
    "font.family":       "monospace",
})

## 1. Load data

In [ ]:
ds = noawclg.load(
    var="prate",
    run_date="20260418",
    cycle="00",
    forecast_hours=[0],
)

lat   = ds["lat"].values
lon   = ds["lon"].values

# prate: kg m⁻² s⁻¹ → mm/h
field = ds["prate"].isel(time=0).values * 3600.0

# 0–360 → −180/180
lon_180 = np.where(lon > 180, lon - 360, lon)
order   = np.argsort(lon_180)
lon_180 = lon_180[order]
field   = field[:, order]

date_str = str(ds["prate"].time.values[0])[:10]

## 2. PlateCarrée — South America region

`ccrs.PlateCarree()` is the simplest projection: lat/lon on a flat grid.
Use `ax.set_extent([west, east, south, north])` to zoom in on a region.

In [ ]:
# Named regions — copy-paste the one you need
REGIONS = {
    "south_america": [-85, -30, -60, 15],
    "brazil":        [-75, -28, -34,  6],
    "northeast_br":  [-47, -34, -18, -1],
    "north_br":      [-74, -44,  -5,  5.5],
    "global":        [-180, 180, -85, 85],
}

W, E, S, N = REGIONS["south_america"]  # west, east, south, north

In [ ]:
levels_prate = np.array([0.1, 0.5, 1, 2, 4, 6, 8, 10, 15, 20, 30, 50])
cmap_p       = copy.copy(cmocean.cm.rain)
cmap_p.set_under(alpha=0)
cmap_p.set_bad(alpha=0)
norm_p       = mcolors.BoundaryNorm(levels_prate, ncolors=cmap_p.N, clip=False)

proj = ccrs.PlateCarree()  # central_longitude=0 by default

fig, ax = plt.subplots(
    figsize=(16, 9),
    subplot_kw={"projection": proj},
    facecolor=BG,
)
ax.set_facecolor(BG)

# ── extent ───────────────────────────────────────────────────────────────────
ax.set_extent([W, E, S, N], crs=ccrs.PlateCarree())

# ── features ─────────────────────────────────────────────────────────────────
ax.add_feature(cfeature.OCEAN.with_scale("50m"),  facecolor=OCEAN, zorder=0)
ax.add_feature(cfeature.LAND.with_scale("50m"),   facecolor=LAND,  zorder=0)
ax.add_feature(cfeature.COASTLINE.with_scale("50m"),
               edgecolor=COAST, linewidth=0.6, zorder=3)
ax.add_feature(cfeature.BORDERS.with_scale("50m"),
               edgecolor=BORDER, linewidth=0.4, zorder=3)

# State / province borders (cartopy ships NaturalEarth 10m states)
states = cfeature.NaturalEarthFeature(
    "cultural", "admin_1_states_provinces_lines", "10m",
    facecolor="none", edgecolor=STATE, linewidth=0.3,
)
ax.add_feature(states, zorder=3)

# ── field ─────────────────────────────────────────────────────────────────────
cf = ax.pcolormesh(
    lon_180, lat, field,
    cmap=cmap_p, norm=norm_p,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

# ── gridlines ─────────────────────────────────────────────────────────────────
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.4,
    color="#14181c",
    alpha=0.8,
    linestyle="--",
)
gl.top_labels    = False
gl.right_labels  = False
gl.xlocator      = mticker.FixedLocator(range(-90, -20, 10))
gl.ylocator      = mticker.FixedLocator(range(-60, 20, 10))
gl.xformatter    = LONGITUDE_FORMATTER
gl.yformatter    = LATITUDE_FORMATTER
gl.xlabel_style  = {"color": SUBTEXT, "fontsize": 7}
gl.ylabel_style  = {"color": SUBTEXT, "fontsize": 7}

# ── colorbar ─────────────────────────────────────────────────────────────────
cb = fig.colorbar(cf, ax=ax, orientation="horizontal",
                  pad=0.05, fraction=0.03, shrink=0.8)
cb.set_label("Precipitation Rate (mm/h)", fontsize=8, color=SUBTEXT)
cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb.outline.set_edgecolor(BOXEDGE)

# ── titles ───────────────────────────────────────────────────────────────────
ax.set_title("Precipitation Rate — South America",
             fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax.set_title(f"GFS 00z  {date_str}",
             fontsize=7,  color=SUBTEXT, loc="right", pad=6)

fig.tight_layout()
fig.savefig("plate_prate_sa.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Annotate specific cities

Use `ax.plot()` for the marker and `ax.annotate()` for the label.
Both need `transform=ccrs.PlateCarree()` or `xycoords=ccrs.PlateCarree()._as_mpl_transform(ax)`.

In [ ]:
cities = [
    ("Fortaleza",  -3.72, -38.54),
    ("Brasília",  -15.78, -47.93),
    ("São Paulo", -23.55, -46.63),
    ("Manaus",     -3.10, -60.02),
]

for name, city_lat, city_lon in cities:
    ax.plot(
        city_lon, city_lat,
        marker="o", markersize=5,
        color="#f7c948",
        markeredgecolor=BG, markeredgewidth=0.8,
        transform=ccrs.PlateCarree(),
        zorder=5,
    )
    ax.annotate(
        name,
        xy=(city_lon, city_lat + 0.8),
        xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),
        fontsize=7, color=TEXT,
        fontweight="bold", fontfamily="monospace",
        ha="center", va="bottom",
        bbox=dict(boxstyle="round,pad=0.3",
                  facecolor=BG, alpha=0.6, edgecolor="none"),
        zorder=6,
    )

fig.savefig("plate_prate_sa_annotated.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Robinson World Map

Robinson is a pseudo-cylindrical projection that balances size and shape distortion — ideal for global thematic maps.

Key difference from PlateCarrée: the projection's `central_longitude` shifts the map centre.

In [ ]:
ds_t2m = noawclg.load(var="t2m", run_date="20260418",
                       cycle="00", forecast_hours=[0])
field_t = ds_t2m["t2m"].isel(time=0).values - 273.15
field_t = field_t[:, order]  # reuse the same lon_180 order from above

levels_t = np.arange(-20, 52, 2)
cmap_t   = copy.copy(cmocean.cm.thermal)
cmap_t.set_under(alpha=0); cmap_t.set_bad(alpha=0)
norm_t   = mcolors.BoundaryNorm(levels_t, ncolors=cmap_t.N, clip=False)

In [ ]:
# Central longitude shifts the map — 0 = Greenwich-centred, -60 = Atlantic
CENTRAL_LON_ROB = 0

proj_rob = ccrs.Robinson(central_longitude=CENTRAL_LON_ROB)

fig2, ax2 = plt.subplots(
    figsize=(16, 9),
    subplot_kw={"projection": proj_rob},
    facecolor=BG,
)
ax2.set_facecolor(BG)
ax2.set_global()

ax2.add_feature(cfeature.OCEAN.with_scale("110m"),  facecolor=OCEAN, zorder=0)
ax2.add_feature(cfeature.LAND.with_scale("110m"),   facecolor=LAND,  zorder=0)
ax2.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.5, zorder=3)
ax2.add_feature(cfeature.BORDERS.with_scale("110m"),
                edgecolor=BORDER, linewidth=0.3, zorder=3)

cf2 = ax2.pcolormesh(
    lon_180, lat, field_t,
    cmap=cmap_t, norm=norm_t,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

gl2 = ax2.gridlines(linewidth=0.3, color="#14181c", alpha=0.6, linestyle="--")

cb2 = fig2.colorbar(cf2, ax=ax2, orientation="horizontal",
                    pad=0.04, fraction=0.03, shrink=0.7)
cb2.set_label("2m Temperature (°C)", fontsize=8, color=SUBTEXT)
cb2.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb2.outline.set_edgecolor(BOXEDGE)

ax2.set_title("Global 2m Temperature",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax2.set_title(f"GFS 00z  {date_str}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig2.tight_layout()
fig2.savefig("robinson_t2m.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Exercises

1. **Different region** — change `REGIONS["south_america"]` to `REGIONS["brazil"]` and increase feature resolution to `"10m"`.
2. **Different variable** — load `"r2"` (relative humidity, already in %) and use `cmocean.cm.haline`.
3. **Atlantic-centred Robinson** — set `CENTRAL_LON_ROB = -30` so the Americas and Europe/Africa are centred.
4. **Contour lines** — add `ax.contour()` on top of the pcolormesh for the MSLP isobars.

---
**Next:** [04_nearside_satellite.ipynb](04_nearside_satellite.ipynb)